# 13 — Security, Safe Parsing, and Secrets

In [ ]:
# Setup: run this first. On Google Colab (or anywhere the course files are missing) it
# downloads the course and moves into this notebook's folder. Elsewhere it does nothing.
import os, sys
if not os.path.exists("../README.md"):
    course = os.path.join(os.path.expanduser("~"), "python-notebook")
    if not os.path.isdir(course):
        os.system(f"git clone -q --depth 1 https://github.com/SwarmLadder/python-notebook {course}")
    os.chdir(os.path.join(course, "advanced", "notebooks"))
    sys.path.insert(0, os.getcwd())

In [ ]:
import json

from lesson import Exercises, ___, run

check = Exercises('advanced/13', {1: '52bc29b68545131e', 2: '8d1d4d8bac4302ef', 3: '79ae24a78191b58b', 4: '7289bae43e7d3b25', 5: 'f12969ae64f685bc', 6: '35cd8c64bcb67a8b', 7: 'a59b19cc8f031391', 8: 'cc8852a00f1d6d01', 9: '63992ffa47680c56', 10: 'a201681582185e7b', 11: 'f4e2aecc9ecc6730'}, practice='advanced_03_security_testing_performance', independent='redact')

## 1. Untrusted input

- Parse data with `json.loads`, never `eval`. Never `pickle.load` data you didn't create: it can run code.
- Limit input size before parsing, and check the shape (`isinstance(data, dict)`).
- Resolve user paths and make sure they stay under the base folder (`Path.resolve()` and `is_relative_to`).

In [ ]:
import json
from pathlib import Path

def parse_limited_json(text, max_bytes=100):
    if len(text.encode()) > max_bytes:
        raise ValueError("too large")
    data = json.loads(text)
    if not isinstance(data, dict):
        raise ValueError("expected an object")
    return data

print(parse_limited_json('{"a": 1}'))
base = Path("/srv/files").resolve()
for p in ["report.txt", "../../etc/passwd"]:
    print(p, (base / p).resolve().is_relative_to(base))

**Exercise 1.** Measure the input size in bytes, not characters. How many bytes is `"héllo"` in UTF-8?

In [ ]:
size = len("héllo".encode())
predicted = ___
check(1, predicted)

**Exercise 2.** Accept only JSON objects. Which of these are dicts?

In [ ]:
import json
texts = ['{"a": 1}', "[1, 2]", '"hi"']
ok = [isinstance(json.loads(t), ___) for t in texts]
check(2, ok)

**Exercise 3.** Does `../secret` stay under the base folder?

In [ ]:
from pathlib import Path
base = Path("/srv/files")
inside = (base / "../secret").resolve().is_relative_to(base)
predicted = ___
check(3, predicted)

**Exercise 4.** Reject text longer than the limit.

In [ ]:
def limit_text(text, max_chars=5):
    if len(text) > ___:
        raise ValueError("too long")
    return text

results = []
for t in ["hey", "far too long"]:
    try:
        results.append(limit_text(t))
    except ValueError:
        results.append("rejected")
check(4, results)

## 2. Secrets and passwords

- Use `secrets.token_urlsafe(32)` for tokens; `random` is predictable and not for security.
- Compare secrets with `hmac.compare_digest(a, b)` so timing doesn't leak how much matched.
- Store passwords as a slow salted hash: `hashlib.pbkdf2_hmac("sha256", password.encode(), salt, iterations).hex()`.

In [ ]:
import secrets, hmac, hashlib
token = secrets.token_urlsafe(32)
print(len(token) >= 43, hmac.compare_digest(token, token))
salt = b"fixed-salt-for-demo"
print(hashlib.pbkdf2_hmac("sha256", b"hunter2", salt, 1000).hex()[:16])

**Exercise 5.** Which module makes security tokens: `"random"` or `"secrets"`?

In [ ]:
answer = ___
check(5, answer)

**Exercise 6.** Compare two tokens in constant time.

In [ ]:
import hmac
same = hmac.compare_digest("abc", "" + ___)
check(6, same)   # compare with "abc"

**Exercise 7.** Hash the same password twice with the same salt. Equal?

In [ ]:
import hashlib
h1 = hashlib.pbkdf2_hmac("sha256", b"pw", b"salt", 1000).hex()
h2 = hashlib.pbkdf2_hmac("sha256", b"pw", b"salt", 1000).hex()
predicted = ___
check(7, predicted)

**Exercise 8.** A different salt gives a different hash. Write `True` or `False`.

In [ ]:
answer = ___
check(8, answer)

## 3. Commands and logs

- Run programs with an argument list and no shell: `subprocess.run(["grep", "-n", "--", pattern, path])`.
- `--` ends the options, so a pattern starting with `-` isn't read as a flag.
- Redact sensitive keys (`password`, `token`, `api_key`) before logging, case-insensitively and in nested dicts.

In [ ]:
def redact(record, sensitive=("password", "token", "api_key")):
    out = {}
    for k, v in record.items():
        if isinstance(v, dict):
            out[k] = redact(v, sensitive)
        elif k.lower() in sensitive:
            out[k] = "***"
        else:
            out[k] = v
    return out

print(["grep", "-n", "--", "-v", "notes.txt"])
print(redact({"user": "ana", "Password": "x", "auth": {"token": "t"}}))

**Exercise 9.** Build the argument list for `grep -n -- PATTERN PATH`.

In [ ]:
pattern, path = "-rf", "a.txt"
args = ["grep", "-n", ___, pattern, path]
check(9, args)

**Exercise 10.** Redact keys case-insensitively.

In [ ]:
record = {"Token": "abc", "user": "bo"}
safe = {k: ("***" if k.lower() == ___ else v) for k, v in record.items()}
check(10, safe)

**Exercise 11.** Is `shell=True` with user input safe? Write `True` or `False`.

In [ ]:
answer = ___
check(11, answer)

## Challenges

Real functions, graded by tests. Complete each implementation and run its check. The independent build supplies only the interface.

### 1. `parse_limited_json`

Parse a JSON object safely, with a size limit.

In [ ]:
def parse_limited_json(text: str, max_bytes: int = 10_000) -> dict:
    """Parse a JSON object safely.

    Check the size (UTF-8 bytes) BEFORE parsing; raise ValueError if too big, if
    the JSON is invalid, or if the top level is not an object.
    """
    if ___:
        raise ValueError("payload too large")
    try:
        data = json.loads(text)
    except json.JSONDecodeError as error:
        raise ValueError("invalid JSON") from error
    if ___:
        raise ValueError("expected a JSON object")
    return data

In [ ]:
check('parse_limited_json')

### 2. `make_token`

A URL-safe random token with enough entropy.

In [ ]:
def make_token() -> str:
    """Return a URL-safe random token of at least 32 bytes of entropy (use `secrets`, not `random`)."""
    return ___

In [ ]:
check('make_token')

### 3. `tokens_match`

Compare secrets in constant time.

In [ ]:
def tokens_match(expected: str, provided: str) -> bool:
    """Compare secrets in constant time with hmac.compare_digest."""
    return ___

In [ ]:
check('tokens_match')

### 4. `grep_command`

The argument list for `grep -n -- PATTERN PATH`.

In [ ]:
def grep_command(pattern: str, path: str) -> list[str]:
    """Build an argument list for `grep -n -- PATTERN PATH` (no shell, no string joining).

    The '--' stops a pattern like '-rf' from being read as an option.
    """
    return ___

In [ ]:
check('grep_command')

### Independent build — `redact`

Return a new dictionary with values of password, token, secret, api_key and authorization replaced by '***', matching keys case-insensitively at every dictionary depth. Preserve keys and other values. Leave the input untouched.

Implement the interface below, run its check, and explain one boundary case with a test of your own.

In [ ]:
def redact(record: dict) -> dict:
    raise NotImplementedError

In [ ]:
check('redact')

### 6. `limit_text`

Return short text unchanged; reject long text.

In [ ]:
def limit_text(text: str, max_chars: int = 1000) -> str:
    """Return text unchanged if short enough; raise ValueError otherwise."""
    if ___:
        raise ValueError(f"input longer than {max_chars} characters")
    return text

In [ ]:
check('limit_text')

### 7. `hash_password`

PBKDF2-HMAC-SHA256 of the password, as hex.

In [ ]:
def hash_password(password: str, salt: bytes, iterations: int = 100_000) -> str:
    """PBKDF2-HMAC-SHA256 of the password, returned as hex. Hint: hashlib.pbkdf2_hmac."""
    import hashlib
    return ___

In [ ]:
check('hash_password')

### 8. `verify_password`

Recompute and compare in constant time.

In [ ]:
def verify_password(password: str, salt: bytes, expected_hex: str, iterations: int = 100_000) -> bool:
    """Recompute the hash and compare in constant time."""
    return ___

In [ ]:
check('verify_password')

### 9. `safe_join`

Join a user path under base, rejecting escapes.

In [ ]:
def safe_join(base: str, user_path: str) -> str:
    """Join a user-supplied relative path under base and return the resolved absolute path.

    Raise ValueError if the result escapes base ('../etc/passwd', absolute paths, ...).
    """
    from pathlib import Path
    root = Path(base).resolve()
    target = (root / user_path).resolve()
    if ___:
        raise ValueError("path escapes the base directory")
    return ___

In [ ]:
check('safe_join')

## Quick quiz

Replace each `'?'` in the cell below with your choice and run it. The answers are hidden, but each one is checked.

**1.** To parse untrusted data, use…

   - **a)** pickle.loads
   - **b)** json.loads with validation
   - **c)** eval

**2.** Why is unpickling untrusted data dangerous?

   - **a)** It's slow
   - **b)** It can execute arbitrary code
   - **c)** It loses precision

**3.** For security tokens, use…

   - **a)** random.random()
   - **b)** time.time()
   - **c)** secrets.token_urlsafe()

In [ ]:
from quiz import Quiz

quiz = Quiz('advanced/13', {1: '44b8dfa7dc5c', 2: '270787e9b442', 3: 'dd83297c1e84'})
quiz.answer(1, '?')
quiz.answer(2, '?')
quiz.answer(3, '?')

**Next:** [14 — Advanced Testing: Properties, Integration, Fakes, and CI](14_testing_property_integration.ipynb)